In [7]:
import sys
sys.path.append("../src")

from pyspark.sql import SparkSession, functions as F
import main, m1

spark = (SparkSession.builder
         .master("local[*]")
         .appName("dev-m3")
         .config("spark.driver.memory", "2g")
         .getOrCreate())

trips = main.load_trips(spark, "../databases", [2025], 1.0)       
clean = m1.clean(m1.flag_invalid(trips, years=[2025]))                             
zones = main.load_zones(spark, "../databases/taxi_zone_lookup.csv")  

print(clean.count())
zones.show(5)

[Stage 61:=====================================================>  (23 + 1) / 24]

8482202
+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [8]:
zone_info = zones.select(F.col("LocationID").alias("PULocationID"),
                         F.col("Zone").alias("zone"),
                         F.col("Borough").alias("borough"))

monthly = (clean
    .withColumn("year", F.year("tpep_pickup_datetime"))       
    .withColumn("month", F.month("tpep_pickup_datetime"))     
    .groupBy("PULocationID", "year", "month")                 
    .agg(F.count("*").alias("trips"),                         
         F.avg("fare_amount").alias("avg_fare"),             
         F.stddev("fare_amount").alias("std_fare"),          
         F.avg("trip_distance").alias("avg_distance_mi"),    
         F.stddev("trip_distance").alias("std_distance_mi"))  
    .join(zone_info, "PULocationID"))                         

monthly.orderBy(F.desc("trips")).show(5)

monthly.groupBy("year", "month").agg(F.sum("trips").alias("trips")).orderBy("year", "month").show()

+------------+----+-----+------+------------------+------------------+------------------+------------------+--------------------+---------+
|PULocationID|year|month| trips|          avg_fare|          std_fare|   avg_distance_mi|   std_distance_mi|                zone|  borough|
+------------+----+-----+------+------------------+------------------+------------------+------------------+--------------------+---------+
|         161|2025|    3|154872|15.762292796632064|12.122949357579055|2.3158199028875663| 2.823844037504088|      Midtown Center|Manhattan|
|         237|2025|    3|150129|12.660886637491718| 8.799006087598432|1.7731741369089227|2.0379028095933753|Upper East Side S...|Manhattan|
|         237|2025|    1|147550| 11.97543368349703| 7.952058244045926| 1.659676448661462| 5.127420938064656|Upper East Side S...|Manhattan|
|         161|2025|    1|146192|14.910250629309417|11.165895777110356|2.1853847679763643|2.6197275684500823|      Midtown Center|Manhattan|
|         132|2025| 

[Stage 70:=====================================================>  (23 + 1) / 24]

+----+-----+-------+
|year|month|  trips|
+----+-----+-------+
|2025|    1|2791191|
|2025|    2|2635260|
|2025|    3|3055749|
|2025|    4|      2|
+----+-----+-------+



In [9]:
from pyspark.sql import Window

by_borough_month = Window.partitionBy("borough", "year", "month").orderBy(F.desc("trips"))

ranked = monthly.withColumn("volume_rank_in_borough", F.rank().over(by_borough_month))

(ranked
    .filter("borough = 'Queens' AND month = 1")
    .orderBy("volume_rank_in_borough")
    .select("zone", "trips", "volume_rank_in_borough")
    .show(5))

[Stage 77:============================================>           (19 + 5) / 24]

+-----------------+------+----------------------+
|             zone| trips|volume_rank_in_borough|
+-----------------+------+----------------------+
|      JFK Airport|128042|                     1|
|LaGuardia Airport| 84067|                     2|
|    East Elmhurst|  8960|                     3|
|        Sunnyside|  1632|                     4|
|          Astoria|  1077|                     5|
+-----------------+------+----------------------+
only showing top 5 rows


In [10]:
ranked = ranked.withColumn("month_idx", F.col("year") * 12 + F.col("month"))
zone_3m = Window.partitionBy("PULocationID").orderBy("month_idx").rangeBetween(-2,0)
borough_3m = Window.partitionBy("borough").orderBy("month_idx").rangeBetween(-2,0)

shares = ranked.withColumn("rolling_3m_share",
                           F.sum("trips").over(zone_3m) / F.sum("trips").over(borough_3m))
                                                                    
(shares
     .filter("PULocationID = 132")
     .orderBy("month")
     .select("zone", "month", "trips", "rolling_3m_share")
     .show())

(shares
     .groupBy("borough", "month")
     .agg(F.sum("rolling_3m_share").alias("soma"))
     .orderBy("borough", "month")
     .show(20))

+-----------+-----+------+------------------+
|       zone|month| trips|  rolling_3m_share|
+-----------+-----+------+------------------+
|JFK Airport|    1|128042|0.5392604447439353|
|JFK Airport|    2|109234|0.5269825476286718|
|JFK Airport|    3|141072|0.5228978937475814|
|JFK Airport|    4|     1|0.5149067824677395|
+-----------+-----+------+------------------+



[Stage 95:=====================================================>  (23 + 1) / 24]

+-------------+-----+--------------------+
|      borough|month|                soma|
+-------------+-----+--------------------+
|        Bronx|    1|                 1.0|
|        Bronx|    2|  0.9999999999999999|
|        Bronx|    3|  0.9999999999999999|
|     Brooklyn|    1|  0.9999999999999998|
|     Brooklyn|    2|                 1.0|
|     Brooklyn|    3|  0.9999999999999998|
|          EWR|    1|                 1.0|
|          EWR|    2|                 1.0|
|          EWR|    3|                 1.0|
|    Manhattan|    1|  1.0000000000000002|
|    Manhattan|    2|  0.9999999999999999|
|    Manhattan|    3|  1.0000000000000002|
|    Manhattan|    4|0.016285900028022476|
|       Queens|    1|                 1.0|
|       Queens|    2|  0.9999933370941736|
|       Queens|    3|                 1.0|
|       Queens|    4|  0.5149067824677395|
|Staten Island|    1|  0.9999999999999996|
|Staten Island|    2|  0.9672131147540984|
|Staten Island|    3|  0.9682539682539684|
+----------

In [14]:
deviation_sigmas = 1.5

zone_prev_3m = Window.partitionBy("PULocationID").orderBy("month_idx").rangeBetween(-3, -1)

flagged = (shares
    .withColumn("prev_3m_mean_trips", F.avg("trips").over(zone_prev_3m))
    .withColumn("prev_3m_std_trips", F.stddev("trips").over(zone_prev_3m))
    .withColumn("volume_anomaly", F.coalesce(
        F.abs(F.col("trips") - F.col("prev_3m_mean_trips")) > deviation_sigmas * F.col("prev_3m_std_trips"),
        F.lit(False))))

(flagged
    .filter("PULocationID = 132")
    .orderBy("month")
    .select("zone", "month", "trips", "prev_3m_mean_trips", "prev_3m_std_trips", "volume_anomaly")
    .show())

flagged.filter("volume_anomaly").groupBy("year", "month").count().orderBy("year", "month").show()

+-----------+-----+------+------------------+------------------+--------------+
|       zone|month| trips|prev_3m_mean_trips| prev_3m_std_trips|volume_anomaly|
+-----------+-----+------+------------------+------------------+--------------+
|JFK Airport|    1|128042|              NULL|              NULL|         false|
|JFK Airport|    2|109234|          128042.0|              NULL|         false|
|JFK Airport|    3|141072|          118638.0|13299.264340556585|          true|
|JFK Airport|    4|     1|          126116.0|16006.144695085071|          true|
+-----------+-----+------+------------------+------------------+--------------+



[Stage 113:====================================================>  (23 + 1) / 24]

+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2025|    3|  168|
|2025|    4|    2|
+----+-----+-----+

